## DATA LOADING

In [14]:
import pandas as pd
from math import sqrt

import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


excel_file = 'manoufactouring_dowentime_full_data.xlsx' 

df_prod = pd.read_excel(excel_file, sheet_name='synthetic_line_productivity')
df_products = pd.read_excel(excel_file, sheet_name='synthetic_products')
df_downtime = pd.read_excel(excel_file, sheet_name='synthetic_line_downtime')
df_factors = pd.read_excel(excel_file, sheet_name='synthetic_downtime_factors')

## ANALYTICAL VARIABLE PREPARATION

In [16]:
df_master = pd.merge(df_prod, df_products[['Product', 'Flavor', 'Min batch time']], on='Product', how='left')
df_master = pd.merge(df_master, df_downtime, on='Batch', how='left')

In [18]:
df_master = pd.merge(df_prod, df_products[['Product', 'Flavor', 'Min batch time']], on='Product', how='left')
df_master = pd.merge(df_master, df_downtime, on='Batch', how='left')

In [20]:
df_master['Start_DT'] = pd.to_datetime(df_master['Date'].astype(str) + ' ' + df_master['Start Time'].astype(str))
df_master['End_DT'] = pd.to_datetime(df_master['Date'].astype(str) + ' ' + df_master['End Time'].astype(str))


In [22]:
df_master['Start_DT'] = pd.to_datetime(df_master['Date'].astype(str) + ' ' + df_master['Start Time'].astype(str))
df_master['End_DT'] = pd.to_datetime(df_master['Date'].astype(str) + ' ' + df_master['End Time'].astype(str))

df_master.loc[df_master['End_DT'] < df_master['Start_DT'], 'End_DT'] += pd.Timedelta(days=1)
df_master['duration_minutes'] = (df_master['End_DT'] - df_master['Start_DT']).dt.total_seconds() / 60

downtime_cols = [str(i) for i in range(1, 13)]
df_master['total_downtime_min'] = df_master[downtime_cols].sum(axis=1)
df_master['net_production_min'] = df_master['duration_minutes'] - df_master['total_downtime_min']

df_master['efficiency_ratio'] = df_master['Min batch time'] / df_master['net_production_min']
df_master['efficiency_ratio'] = df_master['efficiency_ratio'].replace([np.inf, -np.inf], np.nan) 
df_master.dropna(subset=['efficiency_ratio'], inplace=True)
df_master['Week'] = df_master['Start_DT'].dt.isocalendar().week.astype(int)

op_err_factors = df_factors[df_factors['Operator Error'] == 'Yes']['Factor'].tolist()
op_err_cols = [str(f) for f in op_err_factors]
non_op_err_cols = [str(f) for f in df_factors[df_factors['Operator Error'] == 'No']['Factor'].tolist()]
df_master['operator_error_downtime'] = df_master[op_err_cols].sum(axis=1)

## ANSWERING QUERIES AND FORCASTING

In [68]:
most_common_non_op_factor = df_master[non_op_err_cols].sum().idxmax()
factor_desc = df_factors[df_factors['Factor'] == int(most_common_non_op_factor)]['Description'].iloc[0]

contingency_table = pd.crosstab(df_master['Operator'], df_master[most_common_non_op_factor].gt(0))
chi2, p, _, _ = stats.chi2_contingency(contingency_table)

df_master['week_num'] = df_master['Start_DT'].dt.isocalendar().week.astype(int)
weekly_factor = df_master.groupby('week_num')[most_common_non_op_factor].sum().reset_index()
weekly_factor.rename(columns={most_common_non_op_factor: 'factor_downtime'}, inplace=True)

weekly_factor.dropna(inplace=True)

X = weekly_factor[['week_num']]
y = weekly_factor['factor_downtime']

model = LinearRegression()
model.fit(X, y)
future_weeks = np.arange(weekly_factor['week_num'].max() + 1,
                         weekly_factor['week_num'].max() + 5).reshape(-1, 1)

future_weeks_df = pd.DataFrame(future_weeks, columns=['week_num'])

future_predictions = model.predict(future_weeks_df)


print("\n--- Forecasting of Most Common Non-Operator Factor ---")
print(f"Factor {most_common_non_op_factor} ({factor_desc})")
print("\nNext 4 Weeks Forecast:")
for w, pred in zip(future_weeks.flatten(), future_predictions):
    print(f"Week {w}: Expected Downtime = {pred:.2f} minutes")



--- Forecasting of Most Common Non-Operator Factor ---
Factor 3 (Labeling error)

Next 4 Weeks Forecast:
Week 12: Expected Downtime = 200.95 minutes
Week 13: Expected Downtime = 206.94 minutes
Week 14: Expected Downtime = 212.93 minutes
Week 15: Expected Downtime = 218.92 minutes


In [50]:

group_with_f7 = df_master[df_master['7'].gt(0)]['6']
group_without_f7 = df_master[df_master['7'].eq(0)]['6']
f6_mean_with = group_with_f7.mean()
t_stat, p_value_t = stats.ttest_ind(group_with_f7, group_without_f7, equal_var=False, nan_policy='omit')
q2_conclusion = 'Significant difference (P < 0.05)' if p_value_t < 0.05 else 'No significant difference (P > 0.05)'
print(f"\n2. Sequential Effect (F7 'Failure' -> F6 'Adjustment'):\n- Avg F6 when F7 occurred: {f6_mean_with:.2f} min.\n- Conclusion: {q2_conclusion}")

df_master['week_num'] = df_master['Start_DT'].dt.isocalendar().week.astype(int)
weekly_factor = df_master.groupby('week_num')['6'].sum().reset_index()
weekly_factor.rename(columns={'6': 'f6_downtime'}, inplace=True)
weekly_factor.dropna(inplace=True)

X = weekly_factor[['week_num']]
y = weekly_factor['f6_downtime']

model = LinearRegression()
model.fit(X, y)

future_weeks = pd.DataFrame(
    np.arange(weekly_factor['week_num'].max() + 1, weekly_factor['week_num'].max() + 5),
    columns=['week_num']
)

future_predictions = model.predict(future_weeks)

y_pred = model.predict(X)
mae = mean_absolute_error(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print("\n--- Forecasting of F6 Downtime ---")
print("\nNext 4 Weeks Forecast:")
for w, pred in zip(future_weeks['week_num'], future_predictions):
    print(f"Week {w}: Expected Downtime = {pred:.2f} minutes")


2. Sequential Effect (F7 'Failure' -> F6 'Adjustment'):
- Avg F6 when F7 occurred: 2.15 min.
- Conclusion: No significant difference (P > 0.05)

--- Forecasting of F6 Downtime ---

Next 4 Weeks Forecast:
Week 12: Expected Downtime = 62.25 minutes
Week 13: Expected Downtime = 52.46 minutes
Week 14: Expected Downtime = 42.67 minutes
Week 15: Expected Downtime = 32.88 minutes


In [65]:
weekly_downtime = df_master.groupby('Week').agg(
    total_downtime=('total_downtime_min', 'sum'),
    total_op_error=('operator_error_downtime', 'sum')
).reset_index()

weekly_downtime['op_error_proportion'] = weekly_downtime['total_op_error'] / weekly_downtime['total_downtime']
weekly_downtime.replace([np.inf, -np.inf], np.nan, inplace=True)
weekly_downtime.dropna(subset=['op_error_proportion'], inplace=True)

X = weekly_downtime[['Week']]
y = weekly_downtime['op_error_proportion']

model = LinearRegression()
model.fit(X, y)

future_weeks = pd.DataFrame(
    np.arange(weekly_downtime['Week'].max() + 1, weekly_downtime['Week'].max() + 5),
    columns=['Week']
)

future_predictions = model.predict(future_weeks)


print("\n--- Forecasting Operator Error Proportion ---")
print("\nNext 4 Weeks Forecast:")
for w, pred in zip(future_weeks['Week'], future_predictions):
    print(f"Week {w}: Expected Proportion = {pred:.4f}")


--- Forecasting Operator Error Proportion ---

Next 4 Weeks Forecast:
Week 12: Expected Proportion = 0.5860
Week 13: Expected Proportion = 0.5787
Week 14: Expected Proportion = 0.5713
Week 15: Expected Proportion = 0.5640


In [70]:
perfect_batches = df_master[df_master['total_downtime_min'] == 0]

perfect_batches.loc[:, 'week_num'] = perfect_batches['Start_DT'].dt.isocalendar().week.astype(int)

weekly_avg_runtime = perfect_batches.groupby('week_num')['net_production_min'].mean().reset_index()

X = weekly_avg_runtime[['week_num']]
y = weekly_avg_runtime['net_production_min']

model = LinearRegression()
model.fit(X, y)

future_weeks = pd.DataFrame(
    np.arange(weekly_avg_runtime['week_num'].max() + 1, weekly_avg_runtime['week_num'].max() + 5),
    columns=['week_num']
)

future_predictions = model.predict(future_weeks)


print("\n--- Forecasting Average Net Production Time for Perfect Batches ---")
print("\nNext 4 Weeks Forecast:")
for w, pred in zip(future_weeks['week_num'], future_predictions):
    print(f"Week {w}: Expected Net Production Time = {pred:.2f} minutes")


--- Forecasting Average Net Production Time for Perfect Batches ---

Next 4 Weeks Forecast:
Week 12: Expected Net Production Time = 66.91 minutes
Week 13: Expected Net Production Time = 66.97 minutes
Week 14: Expected Net Production Time = 67.04 minutes
Week 15: Expected Net Production Time = 67.10 minutes
